In [ ]:
#packages

using Pkg
#Pkg.add("LinearAlgebra")
#Pkg.add("LinearSolve")
#Pkg.add("Plots")
#Pkg.add("Printf")
#Pkg.add("LaTeXStrings")
#Pkg.add("Interpolations")
#Pkg.add("NPZ")


using LinearAlgebra
using LinearSolve
using Plots
using LaTeXStrings
using Interpolations
using NPZ
using Printf


#-------------------------------------------------------------------------------------
#	dressing of function f (given as array f_discr, occupation ratio given as n_discr)
#-------------------------------------------------------------------------------------


function f_varphimat(lam_discr,gb)
    L = size(lam_discr)[1]
    varphi(lam) = 2*gb/( gb^2 + lam^2 )
    varphimat = zeros(L,L)
    

    for i = 1:L
        for j = 1:i
            varphimat[i,j] = varphi( lam_discr[i]-lam_discr[j] )
            varphimat[j,i] = varphi( lam_discr[i]-lam_discr[j] )
        end
    end
    varphimat = Symmetric(varphimat)
    return varphimat
end 

function f_dlam(lam_discr)
    L = size(lam_discr)[1]
    dlam = zeros(L)
    for i = 2:L-1
        dlam[i] = 0.5*(lam_discr[i+1]-lam_discr[i-1])
    end
    dlam[1] = 0.5*(lam_discr[2]-lam_discr[1])
    dlam[L] = 0.5*(lam_discr[L]-lam_discr[L-1])
    return dlam
end
    
function dress(gb, lam_discr, n_discr, f_discr)
    
    
    varphimat = f_varphimat(lam_discr,gb)
    dlam = f_dlam(lam_discr)
    
    A = I - 0.5/pi * varphimat*Diagonal(n_discr)*Diagonal(dlam)
    A\f_discr
end

#-------------------------------------------------------------------------------------
#	to evaluate charge density associated with function f
#-------------------------------------------------------------------------------------

function charge_density(gb, lam_discr, n_discr, f_discr)
    L = size(lam_discr)[1]
    varphi(lam) = 2*gb/( gb^2 + lam^2 )
    varphimat = zeros(L,L)
    dlam = zeros(L)

    for i = 1:L
        for j = 1:i
            varphimat[i,j] = varphi( lam_discr[i]-lam_discr[j] )
            varphimat[j,i] = varphi( lam_discr[i]-lam_discr[j] )
        end
    end
    varphimat = Symmetric(varphimat)

    for i = 2:L-1
        dlam[i] = 0.5*(lam_discr[i+1]-lam_discr[i-1])
    end
    dlam[1] = 0.5*(lam_discr[2]-lam_discr[1])
    dlam[L] = 0.5*(lam_discr[L]-lam_discr[L-1])
    A = I - 0.5/pi * varphimat*Diagonal(n_discr)*Diagonal(dlam)
    0.5/pi * dot(dlam , Diagonal(n_discr) * (A \ f_discr) )
end

#-------------------------------------------------------------------------
#	solve Yang-Yang equation
#-------------------------------------------------------------------------

function fun1(z)
    if z>0
        return log(1. +exp(-z))
    else
        return log(1. +exp(z)) - z
    end
end

function fun2(z)
    if z<0
        return 1. /(1. +exp(z))
    else
        return exp(-z)/(1. +exp(-z))
    end
end

function yangyang(gb, beta, lam_discr)
    L = size(lam_discr)[1]
    varphi(lam) = 2*gb/( gb^2 + lam^2 )
    varphimat = zeros(L,L)
    dlam = zeros(L)

    for i = 1:L
        for j = 1:i
            varphimat[i,j] = varphi( lam_discr[i]-lam_discr[j] )
            varphimat[j,i] = varphi( lam_discr[i]-lam_discr[j] )
        end
    end
    varphimat = Symmetric(varphimat)

    for i = 2:L-1
        dlam[i] = 0.5*(lam_discr[i+1]-lam_discr[i-1])
    end
    dlam[1] = 0.5*(lam_discr[2]-lam_discr[1])
    dlam[L] = 0.5*(lam_discr[L]-lam_discr[L-1])

    bare_E = 0.5 * beta[3] * lam_discr.^2 + beta[2] * lam_discr + beta[1] * ones(L)
    eps = copy(bare_E)
    n = fun2.(eps)

    #iterative solution
    diff = 1.
    while diff>1e-12
        old_n = copy(n)
        eps = bare_E - 0.5/pi * varphimat*Diagonal( fun1.(eps) )*dlam
        n = fun2.(eps)
        diff = norm(n-old_n)
    end
    return n
end

#-------------------------------------------------------------------------------------
#	compute effective velocity
#-------------------------------------------------------------------------------------

function veff(gb, lam_discr, n_discr)
    L = size(lam_discr)[1]
    varphi(lam) = 2*gb/( gb^2 + lam^2 )
    varphimat = zeros(L,L)
    dlam = zeros(L)

    for i = 1:L
        for j = 1:i
            varphimat[i,j] = varphi( lam_discr[i]-lam_discr[j] )
            varphimat[j,i] = varphi( lam_discr[i]-lam_discr[j] )
        end
    end
    varphimat = Symmetric(varphimat)

    for i = 2:L-1
        dlam[i] = 0.5*(lam_discr[i+1]-lam_discr[i-1])
    end
    dlam[1] = 0.5*(lam_discr[2]-lam_discr[1])
    dlam[L] = 0.5*(lam_discr[L]-lam_discr[L-1])
    dresser = I - 0.5/pi * varphimat*Diagonal(n_discr)*Diagonal(dlam)
    onedr = dresser\ones(L)
    iddr = dresser\copy(lam_discr)
    iddr ./ onedr
end

#-------------------------------------------------------------------------------------
#	time evolution for box expansion
#-------------------------------------------------------------------------------------

function evol_box_expansion(dt, x1, x2, gb, theta_tab, n_fun)
    npts_integral = 600
    L = size(x1)[1]
    
    veff1 = zeros(L)
    veff2 = zeros(L)
    #println(@sprintf "size x2 = %.3f , size theta_tab  = %.3f " size(x1)[1] size(theta_tab)[1])
    #display(x2)
    #display(theta_tab)
    f2 = linear_interpolation(x2, theta_tab)
    f1 = linear_interpolation(x1, theta_tab)
    #velocity of first and last points is the one of free particles
    veff1[1] = theta_tab[1]
    veff2[L] = theta_tab[L]

    for j in 2:L
        thet1 = theta_tab[j]
        if x1[j]<x2[1]
            thet2 = theta_tab[1]
        else
            thet2 = f2(x1[j])
        end
        lam_discr = LinRange(thet2,thet1,npts_integral)
        n_discr = n_fun.(lam_discr)
        veff1[j] = last(veff(gb, lam_discr, n_discr))
        #velocities of right contour (using parity symmetry)
        #veff2[L+1-j] = -veff1[j]
    end
    # sans symetrie
    for j in 1:L-1
        thet1 = theta_tab[j]
        if x2[j] > x1[L]
            thet2 = theta_tab[L]
        else 
            thet2 = f1(x2[j])
        end 
        lam_discr = LinRange(thet1, thet2, 600)
        n_discr = n_fun.(lam_discr)
        veff2[j] = veff(gb, lam_discr, n_discr)[1]
    end
    
    return x1 + dt*veff1, x2 + dt*veff2
end

#-------------------------------------------------------------------------------------
#	box expansion: calculate density (careful: return density in units of rapidities
#    for density in mu^-1 multiply by mass/hbar)
#-------------------------------------------------------------------------------------

function eval_density(x1, x2, gb, theta_tab, n_fun, ratio_m_hbar)
    L = size(x1)[1]
    
    dens = zeros(2,2*L)
    f2 = linear_interpolation(x2, theta_tab)
    f1 = linear_interpolation(x1, theta_tab)
    #density of first and last points is zero
    dens[1,1] = x1[1]
    dens[2,1] = 0
    dens[1,L+1] = x2[L]
    dens[2,L+1] = 0
    #println( @sprintf "n_p = %.3f" ratio_m_hbar * charge_density(gb, theta_tab, n_fun.(theta_tab), ones(size(theta_tab)[1]) ))
    for j in 2:L
        thet1 = theta_tab[j]
        if x1[j]<x2[1]
            thet2 = theta_tab[1]
        else
            #println("nnn")
            thet2 = f2(x1[j])
        end
        lam_discr = LinRange(thet2,thet1,600)
        n_discr = n_fun.(lam_discr)
        dens[1,j] = x1[j]
        dens[2,j] = ratio_m_hbar * charge_density(gb, lam_discr, n_discr, ones(size(lam_discr)[1]) )
        #if j == L 
        #    println( @sprintf " j = %.3f , theta 1 = %.3f , theta 2 = %.3f , n_p1 = %.3f" j thet2 thet1 dens[2,j])
        #end 
        #velocities of right contour (using parity symmetry)
        #dens[1,L+j] = x2[L+1-j]      
        #dens[2,L+j] = dens[2,j]
    end
    #without symmetry 
    for j in 1:L-1
        thet1 = theta_tab[j]
        if x2[j]>x1[L]
            thet2 = theta_tab[L]
        else 
            #println("nnn")
            thet2 = f1(x2[j])
        end 
        lam_discr = LinRange(thet1,thet2,600)
        n_discr = n_fun.(lam_discr)
        dens[1,2*L+1-j] = x2[j] 
        dens[2,2*L+1-j] = ratio_m_hbar * charge_density(gb, lam_discr, n_discr, ones(size(lam_discr)[1]) ) 
        #if j == 1 
        #    println( @sprintf " j = %.3f , theta 1 = %.3f , theta 2 = %.3f , n_p1 = %.3f" j thet1 thet2 dens[2,2*L+1-j])
        #end 
    end
    #sort densities according to positions, and return
    dens[:,sortperm(dens[1, :])]
end

#------------------------------------------------------------------------------------------------
#	function for simulating box expansion. Writes results in file 'data/density_expansion_t.npd'
#     where 't' is the time (takes a list of times to be saved as argument)
#------------------------------------------------------------------------------------------------

function box_expansion(n_fun, dt, list_t_save, theta_max, npts, gb, ratio_m_hbar)
    tmax = maximum(list_t_save)
    theta_tab = LinRange(-theta_max, theta_max, npts)
    x1 = -0.5 * ones(npts) + 0.0000001*theta_tab
    x2 = 0.5 * ones(npts) + 0.0000001*theta_tab
        
    t = 0
    while t<=tmax
        x1, x2 = evol_box_expansion(dt, x1, x2, gb, theta_tab, n_fun)
        t += dt
        
        for ts in list_t_save
            if t-dt<ts && ts<=t
                mydensity = eval_density(x1, x2, gb, theta_tab, n_fun, ratio_m_hbar)
                println(@sprintf "t = %.3f saved" t)
                #filename = @sprintf "%s/density_expansion_n_theta_etoile_conv_py_%.3f_%.3f_%.3f_%.3f.npz" new_date Float64(mu) Float64(T) Float64(Taille) Float64(t)
                #filename = @sprintf "%s/density_expansion_n_py_%.3f_%.3f_%.3f_%.3f.npz" new_date Float64(mu) Float64(T) Float64(Taille) Float64(t)
                filename = @sprintf "%s/density_expansion_n_theta_etoile_py_%.3f_%.3f_%.3f_%.3f.npz" new_date Float64(mup/kB) Float64(T) Float64(Taille) Float64(t)
                #filename = @sprintf "%s/density_expansion_n_tout_py_%.3f_%.3f_%.3f_%.3f.npz" new_date Float64(mup/kB) Float64(T) Float64(Taille) Float64(t)
                npzwrite(filename, mydensity)
            end
        end
    end
end



In [ ]:
function DeltaMat(theta_discr , gbar )
    L = size(theta_discr)[1]
    varphi(theta) = 2*gbar/( gbar^2 + theta^2 )
    varphimat = zeros(L,L)
    

    for i = 1:L
        for j = 1:i
            varphimat[i,j] = varphi( theta_discr[i]-theta_discr[j] )
            varphimat[j,i] = varphi( theta_discr[i]-theta_discr[j] )
        end
    end
    varphimat = Symmetric(varphimat)
    return varphimat
end 

function Delta(theta_discr)
    L = size(theta_discr)[1]
    dtheta = zeros(L)
        
    for i = 2:L-1
        dtheta[i] = 0.5*(theta_discr[i+1]-theta_discr[i-1])
    end
    dtheta[1] = 0.5*(theta_discr[2]-theta_discr[1])
    dtheta[L] = 0.5*(theta_discr[L]-theta_discr[L-1])
    
    return dtheta
end 

function montrer_les_rho(theta_discr, nu_discr)
    L = length(theta_discr)
    
    varphimat = DeltaMat(theta_discr , gbar)
    dtheta = Delta(theta_discr)
    
    # Plot nu_tilde
    p = plot(xlabel="\$\\tilde{\\theta}\$", ylabel="f", legend=:topright)
    plot!(p , theta_discr , nu_discr, label="\$\\tilde{\\nu}\$")
    # Méthode 0
    rho_s_0 = 0.5 / pi * dress(gbar , theta_discr, nu_discr, ones(L))
    rho_0 = nu_discr .* rho_s_0
    
    # Méthode 1
    Mat_F_1 = I - (1 / (2 * pi)) * Diagonal(dtheta) * Diagonal(nu_discr) * varphimat
    B = (1 / (2 * pi)) * nu_discr
    rho_1 = Mat_F_1 \ B
    rho_s_1 = rho_1 ./ nu_discr

    # Méthode 2
    Mat_F_2 = I - 0.5 / pi * varphimat * Diagonal(nu_discr) * Diagonal(dtheta)
    B = (1 / (2 * pi)) * ones(L)
    rho_s_2 = Mat_F_2 \ B
    rho_2 = nu_discr .* rho_s_2
    
    # Plot résultats de la méthode 0
    plot!(p, theta_discr, rho_s_0, label="Meth 0 : \$\\tilde{\\rho_s}= \\frac{1}{2\\pi}\\\$")
    plot!(p, theta_discr, rho_0, label="Meth 0 : \$\\tilde{\\rho} = \\tilde{\\nu} * \\tilde{\\rho_s}\$")
    
    # Plot résultats de la méthode 1
    plot!(p, theta_discr, rho_1, label="Meth 1 : \$\\tilde{\\rho}= \\frac{\\tilde{\\nu}}{2\\pi} ( 1 + \\tilde{\\Delta} * \\tilde{\\rho}) \$")
    plot!(p, theta_discr, rho_s_1, label="Meth 1 : \$\\tilde{\\rho}_s  = \\frac{\\tilde{\\rho}}{\\tilde{\\nu}}  \$")
    
    # Plot résultats de la méthode 2
    plot!(p, theta_discr, rho_s_2, label="Meth 2 : \$\\tilde{\\rho_s}= \\frac{1}{2\\pi}\$")
    plot!(p, theta_discr, rho_2, label="Meth 2 : \$\\tilde{\\rho} = \\tilde{\\nu} * \\tilde{\\rho_s}\$")

    display(p)
    
    #p1 = plot(xlabel="\$\\tilde{\\theta}\$", ylabel="\$\\Delta\$", legend=:topright)

    # Plot des différences entre les résultats des deux méthodes
    #plot(p1,theta_discr, rho_s_1 - rho_s_2, label="\$ \\Delta \\tilde{\\rho_s}\$")
    #plot(p1,theta_discr, rho_1 - rho_2, label="\$\\Delta \\tilde{\\rho}\$")

    # Légendes et labels
    #display(p1)
end

#-------------------------------------------------------------------------------------
#	theta start
#-------------------------------------------------------------------------------------

using Roots
function find_nu_theta_star_discr( theta_discr , theta_star  , nu_discr )
    return ifelse.(theta_discr .> theta_star, 0, nu_discr)
end 

function find_theta_star(theta_discr, nu_discr ,  gbar)
    
    # Définition de la fonction f_veff_star_theta_star
    f_veff_star_theta_star = theta_star -> linear_interpolation(theta_discr, veff( gbar, theta_discr, find_nu_theta_star_discr(theta_discr, theta_star, nu_discr)))(theta_star)
    
    return fzero(f_veff_star_theta_star, theta_discr[1] , last(theta_discr))
end

In [ ]:
import Pkg
#Pkg.add("Dates")
using Dates

# Formater la date selon le format spécifié
new_date = Dates.format(Dates.today(), "yyyy-mm-dd")

# Creer  le dossier
if !isdir(new_date)
    mkdir(new_date)
end

# Afficher la date formatée
println(new_date)

In [ ]:
new_date = "analyse1_2024-03-06"
# Creer  le dossier
if !isdir(new_date)
    mkdir(new_date)
end

# Afficher la date formatée
println(new_date)

In [ ]:
using Plots
using Printf
using Interpolations

# Définir les constantes
hbar = 1.05457182e-25   # um^2.kg/ms
mass =  1.44e-25        # kg (mass of Rubidium 87)
kB = 1.380649e-26       # um^2.ms^-2.kg.nK^{-1}
a3D = 5.3e-3            # um
om_perp = 2 * pi * 2.56  # kHz (transverse frequency)

# Calculer les constantes dérivées
g = 2 * hbar * a3D * om_perp        # effective 1d repulsion strength
c = mass / hbar^2 * g              # um^{-1}
gbar = g / hbar                   # um.ms^{-1}

In [ ]:
theta_max = 20
npts = 200
theta_discr = LinRange(-theta_max, theta_max , npts) 

#------------------------------------------------------------------------------------------------
#	Fonction pour calculer les nu , rho et rho_s et predire cest fonction apres la coupure 1 
#-
function save_nu_rho_rho_s(new_date , nom , theta_discr, n_p  , mup , T , Taille , gbar )
    if mup == "None" 
       mup = hbar * om_perp * (sqrt(1 + 4 * n_p * a3D) - 1)   #kg um^2.ms^-1
    else 
        n_p = (((mup / (hbar * om_perp)) + 1)^2 - 1) / (4 * a3D)
    end
    # Discrétisation de theta
    beta = [-mup / (kB * T) 0 mass * g^2/(kB * T * hbar^2)]
    beta = [-mup / (kB * T) 0 mass /(kB * T)]

    # Définir la date sous forme de chaîne

    index_last_underscore = findlast(isequal('_'), nom)

    
    filename = @sprintf "%s/theta_discr_%s.npz" new_date nom[index_last_underscore+1:end]
    npzwrite(filename, theta_discr )
    println("saved : ", filename)  

    @time nu_discr = yangyang(gbar, beta, theta_discr)
    println("size : ", size(nu_discr)[1])
    filename = @sprintf "%s/nu_discr_%s_%.3f_%.3f.npz" new_date nom[index_last_underscore+1:end] mup/kB T
    npzwrite(filename, nu_discr )
    println("saved : ", filename)
    montrer_les_rho(theta_discr , nu_discr)

    @time rho_s_discr = 0.5 / pi * mass/hbar*dress(gbar, theta_discr, nu_discr, ones(size(theta_discr)[1]))
    filename = @sprintf "%s/rho_s_discr_%s_%.3f_%.3f.npz" new_date nom[index_last_underscore+1:end] mup/kB T
    npzwrite(filename, rho_s_discr )
    println("saved : ", filename)

    @time rho_discr = nu_discr .* rho_s_discr
    filename = @sprintf "%s/rho_discr_%s_%.3f_%.3f.npz" new_date nom[index_last_underscore+1:end] mup/kB T
    npzwrite(filename, rho_discr )
    println("saved : ", filename)

    @time theta_star = find_theta_star(theta_discr, nu_discr ,gbar)
    filename = @sprintf "%s/%s_%.3f_%.3f.npz" new_date nom mup/kB T
    npzwrite(filename, theta_star )
    println("saved : ", filename)

    @time nu_theta_star_discr = find_nu_theta_star_discr(theta_discr , theta_star  , nu_discr )
    filename = @sprintf "%s/nu_%s_discr_%s_%.3f_%.3f.npz" new_date nom[1:index_last_underscore-1] nom[index_last_underscore+1:end] mup/kB T
    nu_theta_star_discr = Float64.(nu_theta_star_discr)
    npzwrite(filename, nu_theta_star_discr )
    println("saved : ", filename)
    
    f_veff_star_theta_star = theta_star -> linear_interpolation(theta_discr, veff( gbar, theta_discr, ifelse.(theta_discr .> theta_star, 0, nu_discr)))(theta_star)
    println( @sprintf "veff (%.3f) = %.3f" theta_star f_veff_star_theta_star(theta_star))

    
    function f_veff_star_theta_star_2(theta_star, theta_discr, veff, gbar, nu_discr)
        #nu_theta_star_discr = find_nu_theta_star_discr(theta_discr, theta_star, nu_discr)
        nu_theta_star_discr_ast = ifelse.(theta_discr .> theta_star, 0, nu_discr)
        v_eff = veff(gbar, theta_discr, nu_theta_star_discr_ast)
        interpolation = LinearInterpolation(theta_discr, v_eff)
        return interpolation(theta_star)
    end

    # Calculer f_veff_star_theta_star pour chaque élément de theta_star
    #f_veff_star_theta_star_vector = [f_veff_star_theta_star_2(theta_star, theta_discr, veff, gbar, nu_discr) for theta_star in theta_star]
    #display(f_veff_star_theta_star_vector)
    f_veff_star_theta_star_vector_1 = zeros(npts)
    f_veff_star_theta_star_vector_2 = zeros(npts)
    for i = 1:npts
        f_veff_star_theta_star_vector_1[i] = f_veff_star_theta_star(theta_discr[i])
        f_veff_star_theta_star_vector_2[i] = f_veff_star_theta_star_2(theta_discr[i], theta_discr, veff, gbar, nu_discr)
    end
    # Affichage des résultats
    for (theta_star_value, f_value) in zip(theta_discr, f_veff_star_theta_star_vector_2)
        if abs.(f_value)<1 
            println(@sprintf "veff (%.3f) = %.3f" theta_star_value f_value)
        end
    end
    
    
    theta_star = fzero(f_veff_star_theta_star, theta_star-1 , theta_star+1)
    println(@sprintf "theta_star = %.3f " theta_star)



    @time rho_s_theta_star_discr = 0.5 / pi * mass/hbar * dress(gbar, theta_discr, nu_theta_star_discr, ones(size(theta_discr)[1]))
    filename = @sprintf "%s/rho_s_%s_discr_%s_%.3f_%.3f.npz" new_date nom[1:index_last_underscore-1] nom[index_last_underscore+1:end] mup/kB T
    npzwrite(filename, rho_s_theta_star_discr )
    println("saved : ", filename)

    @time rho_theta_star_discr = nu_theta_star_discr .* rho_s_theta_star_discr
    filename = @sprintf "%s/rho_%s_discr_%s_%.3f_%.3f.npz" new_date nom[1:index_last_underscore-1] nom[index_last_underscore+1:end] mup/kB T
    npzwrite(filename, rho_theta_star_discr )
    println("saved : ", filename)
    
    return theta_discr , nu_discr , rho_s_discr , rho_discr , theta_star, nu_theta_star_discr, rho_s_theta_star_discr, rho_theta_star_discr

end 

In [ ]:
using DelimitedFiles

date_donnees = "2024-02-29"

DONNEES = []

if date_donnees == "None"
    fichier_x = "Scan43_coupure2_1ms_x.txt"
    fichier_y = "Scan43_coupure2_1ms_n.txt"
    Donnees = [readdlm("2024-02-05/Donnees/$(fichier_x)") , reverse(readdlm("2024-02-05/Donnees/$(fichier_y)")) * 0 , "2em coupure 1ms"]
    push!(DONNEES, Donnees)
    fichier_x = "Scan43_coupure1_10ms_x.txt"
    fichier_y = "Scan43_coupure1_10ms_n.txt"
    Donnees = [readdlm("2024-02-05/Donnees/$(fichier_x)") , reverse(readdlm("2024-02-05/Donnees/$(fichier_y)")) * 0 , "1er coupure 10 ms"]
    push!(DONNEES, Donnees)
    fichier_x = "Scan43_x.txt"
    fichier_y = "Scan43_n.txt"
    Donnees = [readdlm("2024-02-05/Donnees/$(fichier_x)") , reverse(readdlm("2024-02-05/Donnees/$(fichier_y)")) * 0 , "2em coupure 40 ms"]
    push!(DONNEES, Donnees)
elseif date_donnees == "2024-02-05"
    fichier_x = "Scan43_coupure2_1ms_x.txt"
    fichier_y = "Scan43_coupure2_1ms_n.txt"
    Donnees = [readdlm("$(date_donnees)/Donnees/$(fichier_x)") , reverse(readdlm("$(date_donnees)/Donnees/$(fichier_y)")) , "2em coupure 1ms"]
    push!(DONNEES, Donnees)
    fichier_x = "Scan43_coupure1_10ms_x.txt"
    fichier_y = "Scan43_coupure1_10ms_n.txt"
    Donnees = [readdlm("$(date_donnees)/Donnees/$(fichier_x)") , reverse(readdlm("$(date_donnees)/Donnees/$(fichier_y)")) , "1er coupure 10 ms"]
    push!(DONNEES, Donnees)
    fichier_x = "Scan43_x.txt"
    fichier_y = "Scan43_n.txt"
    Donnees = [readdlm("$(date_donnees)/Donnees/$(fichier_x)") , reverse(readdlm("$(date_donnees)/Donnees/$(fichier_y)")) , "2em coupure 40 ms"]
    push!(DONNEES, Donnees)
elseif date_donnees == "2024-02-09"
    fichier_x = "X_Scan704_18ms.txt"
    fichier_y = "Y_Scan704_18ms.txt"
    Donnees = [-readdlm("$(date_donnees)/Donnees/$(fichier_x)") , readdlm("$(date_donnees)/Donnees/$(fichier_y)") , "Première coupure expansion 18 ms"]
    push!(DONNEES, Donnees)
    fichier_x = "X_Scan696_1ms.txt"
    fichier_y = "Y_Scan696_1ms.txt"
    Donnees = [-readdlm("$(date_donnees)/Donnees/$(fichier_x)") , readdlm("$(date_donnees)/Donnees/$(fichier_y)") , "Seconde coupure sélection 1 ms"]
    push!(DONNEES, Donnees)
    fichier_x = "X_Scan696_50ms.txt"
    fichier_y = "Y_Scan696_50ms.txt"
    Donnees = [-readdlm("$(date_donnees)/Donnees/$(fichier_x)") , readdlm("$(date_donnees)/Donnees/$(fichier_y)") , "Seconde coupure expansion 50 ms"]
    push!(DONNEES, Donnees)
elseif date_donnees == "2024-02-29" 
    fichier_x = "X_coupure1_1ms.txt"
    fichier_y = "Y_coupure1_1ms.txt"
    Donnees = [-readdlm("$(date_donnees)/Donnees/$(fichier_x)") , readdlm("$(date_donnees)/Donnees/$(fichier_y)") , "coupure 1 1 ms"]
    push!(DONNEES, Donnees)
    fichier_x = "X_coupure1_28ms.txt"
    fichier_y = "Y_coupure1_28ms.txt"
    Donnees = [-readdlm("$(date_donnees)/Donnees/$(fichier_x)") , readdlm("$(date_donnees)/Donnees/$(fichier_y)") , "coupure 1 18 ms"]
    push!(DONNEES, Donnees)
    fichier_x = "X_coupure2_1ms.txt"
    fichier_y = "Y_coupure2_1ms.txt"
    Donnees = [-readdlm("$(date_donnees)/Donnees/$(fichier_x)") , readdlm("$(date_donnees)/Donnees/$(fichier_y)") , "coupure 2 1 ms"]
    push!(DONNEES, Donnees)
    fichier_x = "X_coupure2_50ms.txt"
    fichier_y = "Y_coupure2_50ms.txt"
    Donnees = [-readdlm("$(date_donnees)/Donnees/$(fichier_x)") , readdlm("$(date_donnees)/Donnees/$(fichier_y)") , "coupure 2 50 ms"]
    push!(DONNEES, Donnees)
end
                            
                            
                            

# Bord ( 2em coupure homogène)

In [ ]:
function pre_model_fit_homo(new_date , nom , theta_discr, n_p , mup ,  T , Taille , gbar )
    
    function box_expansion(n_fun, dt, list_t_save, theta_max, npts, gb, ratio_m_hbar)
        
        filename0 = @sprintf "%s/density_expansion_nu_julia_%.3f_%.3f_%.3f_" new_date Float64(mup/kB) Float64(T) Float64(Taille)
        filename0 = @sprintf "%s/density_expansion_nu_%s_%.3f_%.3f_%.3f_" new_date nom Float64(mup/kB) Float64(T) Float64(Taille)
    
        tmax = maximum(list_t_save)
        theta_tab = LinRange(-theta_max, theta_max, npts)
        x1 = -0.5 * ones(npts) + 0.0000001*theta_tab
        x2 = 0.5 * ones(npts) + 0.0000001*theta_tab

        t = 0

        mydensity = eval_density(x1, x2, gb, theta_tab, n_fun, ratio_m_hbar)
        println(@sprintf "t = %.3f saved" t*Taille)
        filename = filename0 * @sprintf("%.3f.npz", Float64(t))
        println("saved : ", filename)
        npzwrite(filename, mydensity)

        #p = plot(xlabel="x", ylabel="n", legend=:topright)
        #plot!(p, mydensity[1,:],  mydensity[2,:] )
        #display(p)
        while t<tmax
            x1, x2 = evol_box_expansion(dt, x1, x2, gb, theta_tab, n_fun)
            t += dt

            for ts in list_t_save
                if ts <= t < (ts + dt)
                    mydensity = eval_density(x1, x2, gb, theta_tab, n_fun, ratio_m_hbar)
                    println(@sprintf "t = %.3f saved" t*Taille)
                    filename = filename0 * @sprintf("%.3f.npz", Float64(t*Taille))
                    println("saved : ", filename)
                    npzwrite(filename, mydensity)

                    #p = plot(xlabel="x", ylabel="n", legend=:topright)
                    #plot!(p, mydensity[1,:],  mydensity[2,:] )
                    #display(p)
                end
            end
        end  
        
        p = plot(xlabel="x", ylabel="n", legend=:topright)
        plot!(p, mydensity[1,:],  mydensity[2,:] )
        display(p)
        return mydensity[1,:],  mydensity[2,:]
    end
    
    theta_discr , nu_discr , rho_s_discr , rho_discr , theta_star, nu_theta_star_discr, rho_s_theta_star_discr, rho_theta_star_discr = save_nu_rho_rho_s(new_date , nom , theta_discr, n_p , mup ,  T , Taille , gbar )
    # Créer l'interpolation
    #@time nu_fun = LinearInterpolation(theta_discr, nu_discr)
    @time nu_fun = LinearInterpolation(theta_discr, nu_theta_star_discr)

    # Afficher l'interpolation
    #p = plot(xlabel="theta", ylabel="nu", legend=:topright)
    #plot!(p, theta_discr,  nu_discr )
    #plot!(p, theta_discr, nu_fun(theta_discr))
    #display(p)
    
    temps_sav = [0.1/Taille, 0.5/Taille, 1/Taille, 1.5/Taille, 2/Taille , 2.5/Taille , 3/Taille, 4/Taille, 5/Taille, 6/Taille, 7/Taille , 8/Taille , 9/Taille,  10.0/Taille, 20.0/Taille, 30.0/Taille, 40.0/Taille ]
    temps_sav = [ 3/Taille, 4/Taille, 5/Taille, 6/Taille, 7/Taille , 8/Taille , 9/Taille,  10.0/Taille, 20.0/Taille, 30.0/Taille, 40.0/Taille, 50.0/Taille ]
    dt = 0.9/Taille
    temps_sav = [0.1/Taille, 0.5/Taille, 1/Taille, 1.5/Taille, 2/Taille , 2.5/Taille , 3/Taille, 4/Taille, 5/Taille, 6/Taille, 7/Taille , 8/Taille , 9/Taille,  10.0/Taille, 20.0/Taille, 30.0/Taille, 40.0/Taille , 50.0/Taille  ]
    temps_sav = [ 1/Taille, 2/Taille ,  3/Taille, 4/Taille, 5/Taille, 6/Taille, 7/Taille , 8/Taille , 9/Taille,  10.0/Taille, 20.0/Taille, 30.0/Taille, 40.0/Taille , 50.0/Taille  ]

    #temps_sav = [ 50.0/Taille  ]
    dt = 1/Taille
    return box_expansion(nu_fun, dt ,temps_sav  , theta_max , npts , gbar, mass/hbar)
end

nom = "theta_star_1.0_h"
theta_discr = LinRange(-theta_max, theta_max , npts) 

function model_fit_homo(x , p )
    mup , T = p[1]*kB , p[2]
    n_p = "None"
    println("mup = ", mup/kB)
    println("T = ", T)  
    
    x_simul , y_simul = pre_model_fit_homo(new_date , nom , theta_discr, n_p , mup ,  T , Taille , gbar )
    density_fun = LinearInterpolation(x_simul , y_simul) 
    p = plot(xlabel="xxx", ylabel="n", legend=:topright)
    plot!(p, x,  last(DONNEES)[2])
    plot!(p, x,  density_fun.(x))
    display(p)
    return Float64.(density_fun.(x))
end 

In [ ]:
using LsqFit

x_data = last(DONNEES)[1]./Taille
y_data = last(DONNEES)[2]


# Tracer les données
scatter(x_data, y_data, label="Données", xlabel="x", ylabel="y", legend=:topleft)                                                                        
p = plot(xlabel="x", ylabel="n", legend=:topright)
plot!(p, x_data, y_data )
display(p)
                                                                        
size(x_data)
p0 = [49.0 , 200.0] 
# Bornes des paramètres
lower_bounds = 0.5 .* p0
upper_bounds = 1.5 .* p0
# Afficher les bornes
println("lower_bounds = ", lower_bounds)
println("upper_bounds = ", upper_bounds)
# Effectuer l'ajustement avec bornes
#fit = curve_fit(model_fit_homo, x_data, y_data , p0 , lower=[lower_bounds...], upper=[upper_bounds...])

In [ ]:
using LsqFit
using Plots
using Random  # Importation du module Random pour utiliser MersenneTwister

# Définition de la fonction modèle
function model_fit_homo(x, p)
    mup, T = p
    #println("mup = ", mup/kB)
    #println("T = ", T)  
    # Insérez ici le code correspondant à votre modèle
    y = mup * sin.(x) .+ T * x.^2
    return y
end

# Génération de données d'exemple
x_data = collect(range(0, stop=10, length=50))
y_true = model_fit_homo(x_data, [5.0, 0.1])  # Valeurs vraies des paramètres

# Ajout de bruit gaussien aux données pour simuler des données expérimentales
σ = 0.5  # Écart type du bruit
rng = Random.MersenneTwister(1234)  # Initialisation du générateur de nombres aléatoires
y_data = y_true + randn(rng, length(x_data)) * σ  # Données expérimentales bruitées

# Ajustement des données
p0 = [4.0, 0.05]  # Valeurs initiales des paramètres
fit = curve_fit(model_fit_homo, x_data, y_data, p0)

# Récupération des paramètres ajustés
fit_params = coef(fit)

# Affichage des résultats
println("Paramètres ajustés :", fit_params)

# Tracé des données et de l'ajustement
scatter(x_data, y_data, label="Données bruitées")
plot!(x_data, model_fit_homo(x_data, p0), label="premier aj", color=:black)
plot!(x_data, model_fit_homo(x_data, fit_params), label="Ajustement", color=:red)
plot!(x_data, y_true, label="Valeur vraie", linestyle=:dash)
xlabel!("x")
ylabel!("y")
title!("Ajustement de modèle avec données bruitées")


# Bord (2em coupure ihomogène)

## evolution d'un bord 

In [ ]:
#-------------------------------------------------------------------------------------
#	time evolution for box expansion
#-------------------------------------------------------------------------------------

function evol_edge_expansion(dt, x, gb, theta_tab, n_fun)
    npts_integral = 600
    L = size(x)[1]
    
    veff_tab = zeros(L)

    f = linear_interpolation(x, theta_tab)

    #velocity of first and last points is the one of free particles
    veff_tab[1] = theta_tab[1]

    for j in 2:L
        thet1 = theta_tab[j]
        thet2 = theta_tab[1]
        lam_discr = LinRange(thet2,thet1,npts_integral)
        n_discr = n_fun.(lam_discr)
        #println("vell")
        #@time veff_tab[j] = last(veff(gb, lam_discr, n_discr))
        veff_tab[j] = last(veff(gb, lam_discr, n_discr))
        #velocities of right contour (using parity symmetry)
    end
    
    return x + dt*veff_tab
end

#-------------------------------------------------------------------------------------
#	box expansion: calculate density (careful: return density in units of rapidities
#    for density in mu^-1 multiply by mass/hbar)
#-------------------------------------------------------------------------------------

function eval_edge_density(x, gb, theta_tab, nu_fun, ratio_m_hbar)
    L = size(x)[1]
    #L_t = size(theta_tab)[1]
    dens = zeros(2,L)
    #nu = Vector{Vector{Any}}()
    #nu = zeros(L_t,L)
    #f = linear_interpolation(x, theta_tab)

    #density of first and last points is zero
    dens[1,1] = x[1]
    dens[2,1] = 0
    thet1 = theta_tab[1]
    #for i in 1:L 
    #    theta = theta_tab[i]
    #    if theta > thet1 
    #        nu[i,1] = 0 
    #    else
    #        nu[i,1] = nu_fun(theta)      
    #    end
    #end

    for j in 2:L
        thet1 = theta_tab[j]
        thet2 = theta_tab[1]
        lam_discr = LinRange(thet2,thet1,600)
        nu_discr = nu_fun.(lam_discr)
        dens[1,j] = x[j]
        #println("dens")
        #@time dens[2,j] = ratio_m_hbar * charge_density(gb, lam_discr, nu_discr, ones(size(lam_discr)[1]) )        
        dens[2,j] = ratio_m_hbar * charge_density(gb, lam_discr, nu_discr, ones(size(lam_discr)[1]) )
        #for i in 1:L 
        #    theta = theta_tab[i]
         #   if theta > thet1 
         #       nu[i,j] = 0 
         #   else
         #       nu[i,j] = nu_fun(theta)      
         #   end
        #end
    end

    #sort densities according to positions, and return
    return dens[:,sortperm(dens[1, :])] #, nu
end



In [ ]:
function pre_model_fit_inhomo(new_date , nom,  theta_discr, n_p , mup ,  T , Taille , gbar )
    nom = "theta_edge_1.0_ih1"
    theta_discr , nu_discr , rho_s_discr , rho_discr , theta_star, nu_theta_star_discr, rho_s_theta_star_discr, rho_theta_star_discr = save_nu_rho_rho_s(new_date , nom , theta_discr, n_p , mup ,  T , Taille , gbar )
    
    filename0 = @sprintf "%s/density_expansion_nu_%s_%.3f_%.3f_%.3f_" new_date nom Float64(mup/kB) Float64(T) Float64(Taille)
    filename1 = @sprintf "%s/nu_%s_%.3f_%.3f_%.3f_" new_date nom Float64(mup/kB) Float64(T) Float64(Taille)

    function edge_expansion(n_fun, dt, list_t_save, theta_max, npts, gb, ratio_m_hbar)
        tmax = maximum(list_t_save)
        theta_tab = LinRange(-theta_max, theta_max, npts)
        x = 0.0000001*theta_tab

        t = 0
        mydensity = eval_edge_density(x , gb, theta_tab, n_fun, ratio_m_hbar)
        println(@sprintf "t = %.3f saved" t*Taille)
        filename = filename0 * @sprintf("%.3f.npz", Float64(t))
        println("saved : ", filename)
        npzwrite(filename, mydensity)
        filename = @sprintf "%s/bord_%s_%.3f_%.3f_%.3f_%.3f.npz" new_date nom Float64(mup/kB) Float64(T) Float64(Taille) Float64(t*Taille)
        println("saved : ", filename)
        npzwrite(filename, x)

        #filename = filename1 * @sprintf("%.3f.npz", Float64(t*Taille))
        #println("saved : ", filename)
        #println("Taille de nu : ", size(nu))
        #println("Type de nu : ", typeof(nu))

        #nu = map(x -> Float64(x), nu)
        #npzwrite(filename, map(x -> Float64(x), nu))

        p = plot(xlabel="x", ylabel="n", legend=:topright)
        plot!(p, mydensity[1,:],  mydensity[2,:] )
        display(p)
        while t<=tmax
            x = evol_edge_expansion(dt, x , gb, theta_tab, n_fun)
            t += dt

            for ts in list_t_save
                if ts - 0.0001/Taille <= t < (ts + dt) + 0.0001/Taille
                    mydensity = eval_edge_density(x , gb, theta_tab, n_fun, ratio_m_hbar)
                    println(@sprintf "t = %.3f saved" t*Taille)
                    filename = filename0 * @sprintf("%.3f.npz", Float64(t*Taille))
                    println("saved : ", filename)
                    npzwrite(filename, mydensity)
                    filename = @sprintf "%s/bord_%s_%.3f_%.3f_%.3f_%.3f.npz" new_date nom Float64(mup/kB) Float64(T) Float64(Taille) Float64(t*Taille)
                    println("saved : ", filename)
                    npzwrite(filename, x)

                    #p = plot(xlabel="x", ylabel="n", legend=:topright)
                    #plot!(p, mydensity[1,:],  mydensity[2,:] )
                    #display(p)

                    #filename = filename1 * @sprintf("%.3f.npz", Float64(t*Taille))
                    #println("saved : ", filename)
                    #npzwrite(filename, map(x -> Float64(x), nu))

                end
            end
        end
        p = plot(xlabel="x", ylabel="n", legend=:topright)
        plot!(p, mydensity[1,:],  mydensity[2,:] )
        display(p)
    end

    # Créer l'interpolation
    nu_fun = LinearInterpolation(theta_discr, nu_discr)

    # Afficher l'interpolation
    p = plot(xlabel="theta", ylabel="nu", legend=:topright)
    plot!(p, theta_discr,  nu_discr )
    plot!(p, theta_discr, nu_fun(theta_discr))
    display(p)

    temps_sav = [0.1/Taille, 0.5/Taille, 1/Taille, 1.5/Taille, 2/Taille , 2.5/Taille , 3/Taille, 4/Taille, 5/Taille, 6/Taille, 7/Taille , 8/Taille , 9/Taille,  10.0/Taille, 20.0/Taille, 30.0/Taille, 40.0/Taille ]
    temps_sav = [ 0.1/Taille, 0.5/Taille, 1/Taille, 5/Taille, 7/Taille , 8/Taille , 9/Taille,  10.0/Taille, 18.0/Taille]
    temps_sav = [ 1/Taille, 2/Taille ,  3/Taille, 4/Taille, 5/Taille, 6/Taille, 7/Taille , 8/Taille , 9/Taille,  10.0/Taille, 18/Taille , 28/Taille ]
    #temps_sav = [ 28/Taille]
    dt = 1/Taille
    edge_expansion(nu_fun, dt ,temps_sav  , theta_max , npts , gbar, mass/hbar)
    
    ###Coupure de taile $\ell$ + evolution
    t = 28.0/Taille
    filename0 = @sprintf "%s/density_expansion_nu_%s_%.3f_%.3f_%.3f_" new_date nom Float64(mup/kB) Float64(T) Float64(Taille)
    filename = filename0 * @sprintf("%.3f.npz", Float64(t*Taille))
    #filename = @sprintf "%s/density_expansion_nu_edge_julia_%.3f_%.3f_%.3f_%.3f.npz" new_date Float64(mup/kB) Float64(T) Float64(Taille) Float64(t*Taille)
    density = npzread(filename)


    # on coupe où (x/\ell)^2 > 1/4 
    x1 = density[1,:]
    println( size(x1))
    y1 = density[2,:]
    g , d  = 110 - 10 , 110+10
    p = plot(ylabel="x", xlabel="n", legend=:topright)
    plot!(p,  x1[g:d] , y1[g:d])
    display(p)
    x_fun = LinearInterpolation(theta_discr,x1)
    theta_fun = LinearInterpolation(x1, theta_discr)
    x_squared = x1 .^ 2
    x_squared_fun = LinearInterpolation(theta_discr, x_squared)

    f = LinearInterpolation(theta_discr, x_squared)
    f = theta -> linear_interpolation(theta_discr, x_squared.-1/4)(theta)
    p = plot(ylabel="theta", xlabel="x^2 - 1/4", legend=:topright)
    plot!(p,  theta_discr , f(theta_discr))
    display(p)
    theta_1 = fzero(f, theta_discr[1] , theta_fun(0))
    theta_2 = fzero(f, theta_fun(0) , last(theta_discr))

    x2 = ifelse.(x_squared .> 1/4, 0, x1)

    p = plot(ylabel="theta", xlabel="x", legend=:topright)
    plot!(p,  x1 , theta_discr)
    plot!(p,  x2 , theta_discr)
    vline!(p, [x_fun(theta_1), x_fun(theta_2)], linestyle=:dash, color=:red, linewidth=2)
    hline!(p, [theta_1, theta_2], linestyle=:dash, color=:red, linewidth=2)
    display(p)

    #npts = 200
    theta_discr_2 = LinRange(theta_1, theta_2 , npts)
    max_abs_theta = maximum(abs.(theta_discr_2))
    max_abs_theta = maximum([20 , max_abs_theta])
    theta_discr_gauche = LinRange(-max_abs_theta, theta_1 , npts)
    theta_discr_2 = cat(theta_discr_gauche, theta_discr_2, dims=1)

    nom = "theta_edge_1.0_ih2"
    index_last_underscore = findlast(isequal('_'), nom)
    println(nom[index_last_underscore+1:end])

    ## bord gauche 
    x_2 = cat(x_fun(theta_1) * ones(size(theta_discr_gauche)[1]) , LinRange(x_fun(theta_1), x_fun(theta_2) , npts) , dims = 1 )
    x_2_fun = LinearInterpolation(theta_discr_2, x_2)
    theta_discr_2 = LinRange(theta_discr_2[1], last(theta_discr_2) , npts)
    x_2 = x_2_fun(theta_discr_2)
    filename = @sprintf "%s/theta_discr_%s_%.3f_%.3f_%.3f_%.3f.npz" new_date nom[index_last_underscore+1:end] Float64(mup/kB) Float64(T) Float64(Taille) Float64(t*Taille)
    npzwrite(filename, theta_discr_2 )
    println("saved : ", filename)
    filename = @sprintf "%s/x_%s_%.3f_%.3f_%.3f_%.3f.npz.npz" new_date nom[index_last_underscore+1:end] Float64(mup/kB) Float64(T) Float64(Taille) Float64(t*Taille)
    npzwrite(filename, x_2 )
    println("saved : ", filename)



    p = plot(ylabel="theta", xlabel="x", legend=:topright)
    plot!(p,  x , theta_discr)
    plot!(p,  x_2 , theta_discr_2)
    vline!(p, [x_fun(theta_1), x_fun(theta_2)], linestyle=:dash, color=:red, linewidth=2)
    hline!(p, [theta_1, theta_2], linestyle=:dash, color=:red, linewidth=2)
    display(p)

    p = plot(ylabel="theta", xlabel="x", legend=:topright)
    plot!(p,  x_2 , theta_discr_2)
    vline!(p, [0 , x_fun(theta_star)], linestyle=:dash, color=:red, linewidth=2)
    hline!(p, [theta_fun(0) , theta_star], linestyle=:dash, color=:red, linewidth=2)
    display(p)

    #save_nu_rho_rho_s(new_date , nom , theta_discr_2, n_p , mup , T , Taille , gbar )
    theta_discr , nu_discr , rho_s_discr , rho_discr , theta_star, nu_theta_star_discr, rho_s_theta_star_discr, rho_theta_star_discr = save_nu_rho_rho_s(new_date , nom , theta_discr_2, n_p , mup ,  T , Taille , gbar )


    #println( mup )
    filename0 = @sprintf "%s/density_expansion_nu_julia_%.3f_%.3f_%.3f_" new_date Float64(mup/kB) Float64(T) Float64(Taille)
    filename0 = @sprintf "%s/density_expansion_nu_theta_edge_2_julia_%.3f_%.3f_%.3f_" new_date Float64(mup/kB) Float64(T) Float64(Taille)
    filename0 = @sprintf "%s/density_expansion_nu_%s_%.3f_%.3f_%.3f_" new_date nom Float64(mup/kB) Float64(T) Float64(Taille)


    function box_expansion(n_fun,theta_tab, x, dt, list_t_save,  npts, gb, ratio_m_hbar)
        tmax = maximum(list_t_save)
        x1 = x + 0.0000001*theta_tab
        x2 = 0.5 * ones(npts) + 0.0000001*theta_tab

        t = 0

        mydensity = eval_density(x1, x2, gb, theta_tab, n_fun, ratio_m_hbar)
        println(@sprintf "t = %.3f saved" t*Taille)
        filename = filename0 * @sprintf("%.3f.npz", Float64(t))
        println("saved : ", filename)
        npzwrite(filename, mydensity)
        filename = @sprintf "%s/bord1_%s_%.3f_%.3f_%.3f_%.3f.npz" new_date nom Float64(mup/kB) Float64(T) Float64(Taille) Float64(t*Taille)
        println("saved : ", filename)
        npzwrite(filename, x1)
        filename = @sprintf "%s/bord2_%s_%.3f_%.3f_%.3f_%.3f.npz" new_date nom Float64(mup/kB) Float64(T) Float64(Taille) Float64(t*Taille)
        println("saved : ", filename)
        npzwrite(filename, x2)

        p = plot(xlabel="x", ylabel="n", legend=:topright)
        plot!(p, mydensity[1,:],  mydensity[2,:] )
        display(p)

        println(@sprintf "t = %.3f " t*Taille)
        while t<=tmax
            
            x1, x2 = evol_box_expansion(dt, x1, x2, gb, theta_tab, n_fun)
            t += dt
            
            
            #println(@sprintf "t = %.3f " t*Taille)

            for ts in list_t_save
                if ts - 0.0001/Taille <= t < (ts + dt) + 0.0001/Taille
                    #println( ts , t , ts+dt )
                    mydensity = eval_density(x1, x2, gb, theta_tab, n_fun, ratio_m_hbar)
                    println(@sprintf "t = %.3f saved" t*Taille)
                    filename = filename0 * @sprintf("%.3f.npz", Float64(t*Taille))
                    println("saved : ", filename)
                    npzwrite(filename, mydensity)
                    filename = @sprintf "%s/bord1_%s_%.3f_%.3f_%.3f_%.3f.npz" new_date nom Float64(mup/kB) Float64(T) Float64(Taille) Float64(t*Taille)
                    println("saved : ", filename)
                    npzwrite(filename, x1)
                    filename = @sprintf "%s/bord2_%s_%.3f_%.3f_%.3f_%.3f.npz" new_date nom Float64(mup/kB) Float64(T) Float64(Taille) Float64(t*Taille)
                    println("saved : ", filename)
                    npzwrite(filename, x2)
                    #p = plot(xlabel="x", ylabel="n", legend=:topright)
                    #plot!(p, mydensity[1,:],  mydensity[2,:] )
                    #display(p)
                end
            end
            
        end
        p = plot(xlabel="x", ylabel="n", legend=:topright)
        plot!(p, mydensity[1,:],  mydensity[2,:] )
        display(p)
        
        return mydensity[1,:],  mydensity[2,:]
    end

    p = plot(xlabel="theta", ylabel="nu", legend=:topright)
    plot!(p, theta_discr,  nu_discr )
    plot!(p, theta_discr, nu_fun(theta_discr))
    display(p)

    temps_sav = [0.1/Taille, 0.5/Taille, 1/Taille, 1.5/Taille, 2/Taille , 2.5/Taille , 3/Taille, 4/Taille, 5/Taille, 6/Taille, 7/Taille , 8/Taille , 9/Taille,  10.0/Taille, 20.0/Taille, 30.0/Taille, 40.0/Taille , 50.0/Taille ,51.0/Taille ]
    temps_sav = [ 1/Taille, 2/Taille ,  3/Taille, 4/Taille, 5/Taille, 6/Taille, 7/Taille , 8/Taille , 9/Taille,  10.0/Taille, 20.0/Taille, 30.0/Taille, 40.0/Taille , 50.0/Taille  ]
    #temps_sav = [ 1/Taille, 2/Taille ,  3/Taille, 4/Taille, 5/Taille, 6/Taille, 7/Taille , 8/Taille , 9/Taille,  10.0/Taille, 20.0/Taille, 30.0/Taille, 40.0/Taille , 50.0/Taille , 100.0/Taille ,200.0/Taille,300.0/Taille,400.0/Taille , 500.0/Taille  ]
    #temps_sav = [1/Taille]
    dt = 1/Taille
    return box_expansion(nu_fun, theta_discr_2 , x_2 , dt ,temps_sav  , npts , gbar, mass/hbar)
end 

In [ ]:
function model_fit_inhomo(x::Vector{Float64}, p::Vector{Float64})
    mup, T = p[1]*kB, p[2]
    n_p = "None"
    println("mup = ", mup/kB)
    println("T = ", T)  
    Taille = 54 #mu m 
    theta_discr = LinRange(-theta_max, theta_max, npts) 
    nex_date = "fit"
    x_simul, y_simul = pre_model_fit_inhomo(new_date, nom, theta_discr, n_p, mup, T, Taille, gbar)
    println(x_simul, y_simul)
    density_fun = LinearInterpolation(x_simul, y_simul) 
    p = plot(xlabel="xxx", ylabel="n", legend=:topright)
    println( "lalal" )
    println( size(x))
    densi = density_fun.(x)
    
    println( size(densi))
    plot!(p, x,  y_data)
    plot!(p, x_simul, y_simul)
    plot!(p, x, densi)
    display(p)
    return Float64(density_fun(x))
end 
 

In [ ]:
using LsqFit


x_data = last(DONNEES)[1]./Taille
y_data = last(DONNEES)[2]


# Tracer les données
scatter(x_data, y_data, label="Données", xlabel="x", ylabel="y", legend=:topleft)                                                                        
p = plot(xlabel="x", ylabel="n", legend=:topright)
plot!(p, x_data, y_data )
display(p)
                                                                        
size(x_data)
p0 = [48.0 , 500.0] 
# Bornes des paramètres
lower_bounds = 0.5 .* p0
upper_bounds = 1.5 .* p0
# Afficher les bornes
println("lower_bounds = ", lower_bounds)
println("upper_bounds = ", upper_bounds)
# Effectuer l'ajustement avec bornes
#fit = curve_fit(model_fit_inhomo, x_data, y_data , p0 , lower=[lower_bounds...], upper=[upper_bounds...])

In [ ]:
mup, T = 46*kB, 500
n_p = "None"
println("mup = ", mup/kB)
println("T = ", T)  
Taille = 58 #mu m 
theta_discr = LinRange(-theta_max, theta_max, npts) 
new_date = "analyse1_2024-03-06"
new_date = "analyse2_2024-03-01"
#x_simul, y_simul = pre_model_fit_inhomo(new_date, nom, theta_discr, n_p, mup, T, Taille, gbar)

x_data = last(DONNEES)[1]./Taille
y_data = last(DONNEES)[2]


# Tracer les données
scatter(x_data, y_data, label="Données", xlabel="x", ylabel="y", legend=:topleft)                                                                        
p = plot(xlabel="x", ylabel="n", legend=:topright)
plot!(p, x_data, y_data )
display(p)

In [ ]:
#import Pkg; Pkg.add("PlotlyJS") ;Pkg.add("CSV"); Pkg.add("DataFrames")
using PlotlyJS, CSV, DataFrames

mup = 50 *kB 
T = 300

beta = [-mup / (kB * T) 0 mass /(kB * T)]
@time nu_discr = yangyang(gbar, beta, theta_discr)
@time densite = mass/hbar* charge_density(gbar, theta_discr, yangyang(gbar, beta, theta_discr), ones(size(theta_discr)[1]) ) 
println(densite)

function model_prel(T,mu)
    #println(@sprintf "T = %.3f ,mu = %.3f " T mu)
    mup = mu * kB
    beta = [-mup / (kB * T) 0 mass /(kB * T)]
    nu_discr = yangyang(gbar, beta, theta_discr)
    densite = mass/hbar* charge_density(gbar, theta_discr, nu_discr, ones(size(theta_discr)[1]) )
    #println(densite)
    return densite
end

function model(T,mu)
    
    OutDensite = 0*ones(size(T)[1])
    for i in 1:size(T)[1]
        #println(@sprintf "i =, T = %.3f ,mu = %.3f " i T[i] mu[i])
        OutDensite[i] = model_prel(T[i],mu[i])
        #println("OutDensite = " ,OutDensite )
    end
    
                                                                          
    #p = plot(xlabel="mu", ylabel="densite", legend=:topright)
    #plot!(p, mu, OutDensite ,mode="markers",
    #marker=attr(size=12, line=attr(width=2, color="DarkSlateGrey")))
    #display(p)
    
    #println(OutDensite)
    return OutDensite
    
end

T = LinRange(600, 1000 , 5)

Densite = 51*ones(size(T)[1])

println(" size(T) : " , size(T))
println(" size(D) : " , size(Densite))
println(" Densite : " , Densite)

p0 = 50*ones(size(T)[1])


fit = curve_fit(model, T , Densite , p0 )



In [ ]:


mu = coef(fit)

err =  Densite - model(T,mu); z0 = 0*err;
str = @sprintf "fit err max %.3f e-14" maximum(abs, err) * 10^14
plot(mu,model(T,mu), label=str, lw=2, c=:green, yerror=(z0,err), ribbon=(z0,err), fc=:cyan, fa=0.1)
scatter!(mu, Densite, mc=:red, xlabel="mu", ylabel="n", label="Input data: n = n(mu)", legend=:bottomright)
display(p)

#p1 = plot(xlabel="T", ylabel="mu", legend=:topright)
#plot!(p1, T, mu ) 
#display(p1)

In [ ]:
new_date = "ultime"
# Creer  le dossier
if !isdir(new_date)
    mkdir(new_date)
end

# Afficher la date formatée
println(new_date)

for i in 2:size(T)[1]
    n_p = "None"
    Taille = 58 #mu m 
    theta_discr = LinRange(-theta_max, theta_max, npts)  
    x_simul, y_simul = pre_model_fit_inhomo(new_date, nom, theta_discr, n_p, mu[i]*kB, T[i], Taille, gbar)
end 

In [ ]:
function model_prel_mu(T,mu)
    #println(@sprintf "T = %.3f ,mu = %.3f " T mu)
    mup = mu * kB
    beta = [-mup / (kB * T) 0 mass /(kB * T)]
    nu_discr = yangyang(gbar, beta, theta_discr)
    densite = mass/hbar* charge_density(gbar, theta_discr, nu_discr, ones(size(theta_discr)[1]) )
    #println(densite)
    return densite
end

function model_mu(T,mu)
    
    OutDensite = 0*ones(size(T)[1])
    for i in 1:size(T)[1]
        #println(@sprintf "i =, T = %.3f ,mu = %.3f " i T[i] mu[i])
        OutDensite[i] = model_prel_mu(T[i],mu[i])
        #println("OutDensite = " ,OutDensite )
    end
    
                                                                          
    #p = plot(xlabel="mu", ylabel="densite", legend=:topright)
    #plot!(p, mu, OutDensite ,mode="markers",
    #marker=attr(size=12, line=attr(width=2, color="DarkSlateGrey")))
    #display(p)
    
    #println(OutDensite)
    return OutDensite
    
end

T = LinRange(600, 1000 , 5)

#Densite = 51*ones(size(T)[1])


#p0 = 50*ones(size(T)[1])


#fit = curve_fit(model, T , Densite , p0 )

function trier_x_et_y(x::Vector{T}, y::Vector{T}) where T
    # Obtenir les indices qui trient x en ordre croissant
    indices_tries = sortperm(x)
    
    # Trier x en ordre croissant
    x_tries = x[indices_tries]
    
    # Réorganiser y en conséquence
    y_tries = y[indices_tries]
    
    return x_tries, y_tries
end
;
x_data = last(DONNEES)[1]./Taille .-6/Taille
y_data = last(DONNEES)[2]

new_date = "ultime"
# Creer  le dossier
if !isdir(new_date)
    mkdir(new_date)
end

function model_fit_inhomo_un(x, p)
    T = p[1]
    mup = coef(curve_fit(model_mu, T*ones(1) , 51*ones(1) , 50*ones(1) ))[1]*kB   
    n_p = "None"
    println("mup = ", mup/kB)
    println("T = ", T)  
    Taille = 54 #mu m 
    theta_discr = LinRange(-theta_max, theta_max, npts) 
    x_simul, y_simul = pre_model_fit_inhomo(new_date, nom, theta_discr, n_p, mup, T, Taille, gbar)    
    
    #x_simul,y_simul = Float64.(x) , Float64.(y_data)

    x_simul, y_simul = trier_x_et_y(x_simul[:,1], y_simul[:,1])
    
    function density_fun(x)
        L = size(x)[1]
        y = ones(L)
        fun = LinearInterpolation(x_simul, y_simul)
        for i in 1:L
            if minimum(x_simul) < x[i] < maximum(x_simul) 
                y[i] = fun(x[i])
            else 
                y[i] = 0
            end
        end 
        return y
    end
    #density_fun = LinearInterpolation(x_simul, y_simul) 
    p = plot(xlabel="x/Taille", ylabel="n", legend=:topright)

    densi = density_fun(x)

    plot!(p, x,  y_data, label = "data")
    plot!(p, x_simul, y_simul , label = "simule")
    plot!(p, x, densi , label = "intepol")
    display(p)
    return densi
end 
    
using LsqFit


x_data, y_data = trier_x_et_y(x_data[:,1], y_data[:,1])



# Tracer les données
scatter(x_data, y_data, label="Données", xlabel="x", ylabel="y", legend=:topleft)                                                                        
p = plot(xlabel="x", ylabel="n", legend=:topright)
plot!(p, x_data, y_data )
display(p)
                                                                        
println("size x dat :"  , size(x_data))
p0 = 730.613*ones(1) 
println( "size(p0) : " , size(p0))
# Bornes des paramètres
lower_bounds = 0.5 .* p0.*ones(size(p0)[1])
upper_bounds = 1.5 .* p0.*ones(size(p0)[1])
# Afficher les bornes
println("lower_bounds = ", lower_bounds , size(lower_bounds))
println("upper_bounds = ", upper_bounds, size(upper_bounds))
# Effectuer l'ajustement avec bornes
println("3")
println("size x_data : " , size(x_data))
println("size y_data : " , size(y_data))    
fit = curve_fit(model_fit_inhomo_un, x_data, y_data , p0 )    

In [ ]:
Taille = 54 #mu m 


x_data = last(DONNEES)[1]./Taille .-6
y_data = last(DONNEES)[2]

new_date = "ultime3parametre"
# Creer  le dossier
if !isdir(new_date)
    mkdir(new_date)
end


function model_fit_inhomo_trois(x, p)
    mup , T , Taille = p[1]*kB , p[2] , p[3]  
    n_p = "None"
    Taille = 54 #mu m 
    println("mu = ", mup/kB)
    println("T = ", T)  
    println("Taille = ", Taille)
    theta_discr = LinRange(-theta_max, theta_max, npts) 
    x_simul, y_simul = pre_model_fit_inhomo(new_date, nom, theta_discr, n_p, mup, T, Taille, gbar)    
    
    #x_simul,y_simul = Float64.(x) , Float64.(y_data)

    x_simul, y_simul = trier_x_et_y(x_simul[:,1], y_simul[:,1])
    
    function density_fun(x)
        L = size(x)[1]
        y = ones(L)
        fun = LinearInterpolation(x_simul, y_simul)
        for i in 1:L
            if minimum(x_simul) < x[i] < maximum(x_simul) 
                y[i] = fun(x[i])
            else 
                y[i] = 0
            end
        end 
        return y
    end
    #density_fun = LinearInterpolation(x_simul, y_simul) 
    p = plot(xlabel="x/Taille", ylabel="n", legend=:topright)

    densi = density_fun(x)

    plot!(p, x,  y_data, label = "data")
    plot!(p, x_simul, y_simul , label = "simule")
    plot!(p, x, densi , label = "intepol")
    display(p)
    return densi
end 
    
using LsqFit


x_data, y_data = trier_x_et_y(x_data[:,1], y_data[:,1])



# Tracer les données
scatter(x_data, y_data, label="Données", xlabel="x", ylabel="y", legend=:topleft)                                                                        
p = plot(xlabel="x", ylabel="n", legend=:topright)
plot!(p, x_data, y_data )
display(p)
                                                                        
println("size x dat :"  , size(x_data))
p0 = [55 , 910.0 , 59] 
println( "size(p0) : " , size(p0))
# Bornes des paramètres
lower_bounds = 0.5 .* p0
upper_bounds = 1.5 .* p0
# Afficher les bornes
println("lower_bounds = ", lower_bounds , size(lower_bounds))
println("upper_bounds = ", upper_bounds, size(upper_bounds))
# Effectuer l'ajustement avec bornes
println("3")
println("size x_data : " , size(x_data))
println("size y_data : " , size(y_data))    
fit = curve_fit(model_fit_inhomo_trois, x_data, y_data , p0 )    

In [ ]:
x = [1, 2, 3, 4, 5]
10<max(x)

In [ ]:
import Pkg; Pkg.add("GLM")

using Plots, Random, Distributions, GLM, DataFrames, Printf

# input data
n = 30
x = LinRange(0,1, n) .* 10
e = rand(Normal(0,mean(x)/4),n)
parms = Dict(:b0 => 6, :b1 => 3.2)
y = parms[:b0] .+ parms[:b1] .* x .+ 3*e
data = DataFrame(X=x, Y=y)

# linear regression
ols = lm(@formula(Y ~ 1 + X), data)
a, b = coef(ols)
r = r2(ols)

# plotting
yhat = parms[:b0] .+ parms[:b1] .* x
err = y - yhat; z0 = 0*err;
str = "Y = " * @sprintf("%.2f",a) * " + " * @sprintf("%.2f",b) * " X (r = " * @sprintf("%.2f",r) * ")" 
plot(x,yhat, label=str, lw=2, c=:green, yerror=(z0,err), ribbon=(z0,err), fc=:cyan, fa=0.1)
scatter!(x, y, mc=:red, xlabel="X", ylabel="Y", label="Input data: Y = Y(X)", legend=:bottomright)

In [ ]:
## using Printf
using Glob

function extract_t_times_Taille(date::String, nom::String, triple::Tuple{Float64, Float64, Float64})
    mup, T, Taille = triple
    t_times_Taille_values = Set{Float64}()  # Utiliser un ensemble pour éviter les doublons
    
    # Construire le motif de recherche pour le nom de fichier
    pattern = Regex(@sprintf("density_expansion_nu_%s_%.3f_%.3f_%.3f_([\\d.]+)\\.npz", escape(nom), mup, T, Taille))
    
    # Parcourir les fichiers dans le répertoire spécifié par date
    for filename in readdir(date)
        # Vérifier si le nom du fichier correspond au format attendu
        match = match(pattern, filename)
        if match !== nothing
            t_times_Taille = parse(Float64, match.captures[1])
            push!(t_times_Taille_values, t_times_Taille)
        end
    end
    
    return sort!(collect(t_times_Taille_values))
end

function extract_triples(date::String, nom::String, mup::Union{Float64, Nothing}=nothing, T::Union{Float64, Nothing}=nothing, Taille::Union{Float64, Nothing}=nothing)
    triples = Dict{Tuple{Float64, Float64, Float64}, Vector{Float64}}()
    
    # Parcourir les fichiers dans le répertoire spécifié par date
    for filename in readdir(date)
        # Vérifier si le nom du fichier correspond au format attendu
        match = match(r"density_expansion_nu_$(escape(nom))_([\d.]+)_([\d.]+)_([\d.]+)_([\d.]+)\.npz", filename)
        if match !== nothing
            file_mup = parse(Float64, match.captures[1])
            file_T = parse(Float64, match.captures[2])
            file_Taille = parse(Float64, match.captures[3])
            # Vérifier si les valeurs correspondent aux critères spécifiés
            if (mup === nothing || mup == file_mup) && (T === nothing || T == file_T) && (Taille === nothing || Taille == file_Taille)
                triples[(file_mup, file_T, file_Taille)] = extract_t_times_Taille(date, nom, (file_mup, file_T, file_Taille))
            end
        end
    end
    
    return triples
end


In [ ]:
#Pkg.add("FileIO")
using FileIO

function extract_triples(date::String, nom::String, mup::Union{Nothing, Float64}=nothing, T::Union{Nothing, Float64}=nothing, Taille::Union{Nothing, Int64}=nothing)
    triples = Dict{Tuple{Float64, Float64, Float64}, Vector{Float64}}()

    for filename in readdir(date)
        if occursin("density_expansion_nu_$(nom)", filename)
            file_mup, file_T, file_Taille = parse.(Float64, match(r"_(\d+\.\d+)_", filename).captures)
            if mup === nothing || mup == file_mup
                if T === nothing || T == file_T
                    if Taille === nothing || Taille == file_Taille
                        t_times_Taille_values = extract_t_times_Taille(date, nom, (file_mup, file_T, file_Taille))
                        triples[(file_mup, file_T, file_Taille)] = t_times_Taille_values
                    end
                end
            end
        end
    end

    return triples
end

function extract_t_times_Taille(date::String, nom::String, triple::Tuple{Float64, Float64, Float64})
    mup, T, Taille = triple
    t_times_Taille_values = Set{Float64}()

    pattern = r"density_expansion_nu_$(nom)_$(mup)_$(T)_$(Taille)_([\d.]+)\.npz"
    for filename in readdir(date)
        if match(pattern, filename) !== nothing
            t_times_Taille = parse(Float64, match(r"_([\d.]+)\.npz", filename).captures[1])
            push!(t_times_Taille_values, t_times_Taille)
        end
    end

    return sort(collect(t_times_Taille_values))
end


In [ ]:
using FileIO
using Printf

function extract_t_times_Taille(date::String, nom::String, triple::Tuple{Float64, Float64, Float64})
    mup, T, Taille = triple
    t_times_Taille_values = Set{Float64}()

    # Construire le motif de recherche pour le nom de fichier
    pattern = r"density_expansion_nu_$(nom)_$(mup)_$(T)_$(Taille)_([\d.]+)\.npz"
    
    # Parcourir les fichiers dans le répertoire spécifié par date
    for filename in readdir(date)
        # Vérifier si le nom du fichier correspond au format attendu
        match = match(pattern, filename)
        if match !== nothing
            t_times_Taille = parse(Float64, match.captures[1])
            push!(t_times_Taille_values, t_times_Taille)
        end
    end

    return sort(collect(t_times_Taille_values))
end

function extract_triples(date::String, nom::String, mup::Union{Nothing, Float64}=nothing, T::Union{Nothing, Float64}=nothing, Taille::Union{Nothing, Float64}=nothing)
    triples = Dict{Tuple{Float64, Float64, Float64}, Vector{Float64}}()

    # Parcourir les fichiers dans le répertoire spécifié par date
    for filename in readdir(date)
        # Vérifier si le nom du fichier correspond au format attendu
        match = match(r"density_expansion_nu_$(re.escape(nom))_([\d.]+)_([\d.]+)_([\d.]+)_([\d.]+)\.npz", filename)
        if match !== nothing
            file_mup = parse(Float64, match.captures[1])
            file_T = parse(Float64, match.captures[2])
            file_Taille = parse(Float64, match.captures[3])
            # Vérifier si les valeurs correspondent aux critères spécifiés
            if (mup === nothing || mup == file_mup) && (T === nothing || T == file_T) && (Taille === nothing || Taille == file_Taille)
                t_times_Taille_values = extract_t_times_Taille(date, nom, (file_mup, file_T, file_Taille))
                triples[(file_mup, file_T, file_Taille)] = t_times_Taille_values
            end
        end
    end

    return triples
end


In [ ]:
date = "2024-03-01"
nom = "theta_edge_1.0_ih2"
mup = nothing
T = nothing
Taille = 54

# Extraction des triples possibles
triples = extract_triples(date, nom, nothing, nothing, Taille)
println("Triples clés possibles: ", keys(triples))

# Tracé des graphiques pour chaque triple avec les temps max

for (triple, liste_t) in pairs(triples)
    time_list = [maximum(liste_t)]
    println("")
    # Vous devez remplacer la suite du code Python par l'équivalent en Julia pour tracer les graphiques.
    # La création de graphiques avec PyPlot en Julia est similaire à celle en Python, mais les détails peuvent différer légèrement.
end


In [ ]:
using Glob
using Printf
import Base.Iterators: product

function extract_triples(date::String, nom::String, mup::Union{Nothing, Float64}=nothing, T::Union{Nothing, Float64}=nothing, Taille::Union{Nothing, Float64}=nothing)
    triples = Set{Tuple{Float64, Float64, Float64}}()  # Utiliser un ensemble pour éviter les doublons
    
    # Parcourir les fichiers dans le répertoire spécifié par date
    for filename in readdir(joinpath(date))
        # Vérifier si le nom du fichier correspond au format attendu
        m = match(r"density_expansion_nu_$(nom)_([\d.]+)_([\d.]+)_([\d.]+)_([\d.]+)\.npz", filename)
        if m !== nothing
            file_mup = parse(Float64, m.captures[1])
            file_T = parse(Float64, m.captures[2])
            file_Taille = parse(Float64, m.captures[3])
            # Vérifier si les valeurs correspondent aux critères spécifiés
            if (isnothing(mup) || mup == file_mup) && (isnothing(T) || T == file_T) && (isnothing(Taille) || Taille == file_Taille)
                push!(triples, (file_mup, file_T, file_Taille))
            end
        end
    end
    
    return triples
end

function extract_t_times_Taille(date::String, nom::String, triple::Tuple{Float64, Float64, Float64})
    mup, T, Taille = triple
    t_times_Taille_values = Set{Float64}()  # Utiliser un ensemble pour éviter les doublons
    
    # Construire le motif de recherche pour le nom de fichier
    pattern = r"density_expansion_nu_$(nom)_$(mup)_$(T)_$(Taille)_([\d.]+)\.npz"
    
    # Parcourir les fichiers dans le répertoire spécifié par date
    for filename in readdir(joinpath(date))
        # Vérifier si le nom du fichier correspond au format attendu
        m = match(pattern, filename)
        if m !== nothing
            t_times_Taille = parse(Float64, m.captures[1])
            push!(t_times_Taille_values, t_times_Taille)
        end
    end
    
    return t_times_Taille_values
end

date = "2024-02-11"
nom = "theta_star_julia"
mup = nothing
T = nothing
Taille = 42.480

# Extraction des triples possibles
triples = extract_triples(date, nom, mup, T, Taille)
println("Triples possibles:", triples)

for triple in triples
    t_times_Taille_values = extract_t_times_Taille(date, nom, triple)
    println("Temps t*Taille possibles:", t_times_Taille_values)
end


In [ ]:
using Plots
using QuadGK

function integrale( x , y )
# Données discrètes
#x = theta_discr
#y = rho_discr

    # Interpolation des données
    f_interp = LinearInterpolation(x, y)

    # Définir la fonction à intégrer
    function f(x)
        return f_interp(x)
    end

    # Intégration sur un intervalle fini
    integral_value, _ = quadgk(f, minimum(x), maximum(x))

    # Afficher le résultat
    #println("Valeur de l'intégrale sur l'intervalle fini: ", integral_value)
    #p1 = plot(xlabel="npts", ylabel="n_p", legend=:topright)
    # Tracer x par rapport à y
    #plot!(p1 , x, y, xlabel="x", ylabel="y", label="y", seriestype=:scatter)

    # Tracer x par rapport à f(x)
    x_values = range(minimum(x), maximum(x), length=100)
    #plot!(p1, x_values, f_interp(x_values), label="f(x)")
    #display(p1) 
    return integral_value
end

#n_p = 40
#T = 0.1
#liste_npts = LinRange(30, 40, 5)
liste_npts = collect(200:400)
theta_max = 20
n_p_1 = zeros(length(liste_npts))
n_p_2 = zeros(length(liste_npts))

for (i, npts) in enumerate(liste_npts)
    theta_discr = LinRange(-theta_max, theta_max, npts)
    mup = hbar * om_perp * (sqrt(1 + 4 * n_p * a3D) - 1)   #kg um^2.ms^-1
    beta = [-mup / (kB * T) 0 mass / (kB * T)]
    nu_discr = yangyang(gbar, beta, theta_discr)
    rho_s_discr = 0.5 / pi * mass / hbar * dress(gbar, theta_discr, nu_discr, ones(size(theta_discr)[1]))
    rho_discr = nu_discr .* rho_s_discr
    test_1 = mass / hbar * charge_density(gbar, theta_discr, nu_discr, ones(size(theta_discr)[1]))
    #println(test_1)
    n_p_1[i] = test_1
    n_p_2[i] = integrale(theta_discr, rho_discr)
end

p = plot(xlabel="npts", ylabel="n_p", legend=:topright)
plot!(p, liste_npts, n_p_1, label="1")
plot!(p, liste_npts, n_p_2, label="2")
# Ajouter une ligne horizontale pour n_p
hline!(p, [n_p], linestyle=:dash, color=:red, linewidth=2)
display(p)    
    
    

In [ ]:
# Définir un tableau (matrice)
A = [1 2 3 4;
     4 5 6 4;
     7 8 9 4]

# Obtenir la transposée de A en utilisant la fonction transpose()
A_transpose = transpose(A)

# Ou, obtenir la transposée de A en utilisant l'opérateur de transposition '
#A_transpose = A'

# Afficher la matrice A et sa transposée
println("Matrice A : ")
println(A)
println("\nTransposée de A : ")
println(A_transpose)

In [ ]:
function transpose_list_of_lists(list_of_lists::Vector{Vector{T}}) where T
    # Déterminer la longueur de la liste la plus longue dans list_of_lists
    max_length = maximum(length, list_of_lists)

    # Initialiser une liste vide pour stocker la transposée
    transposed_list = Vector{Vector{T}}(undef, max_length)

    # Initialiser chaque élément de transposed_list avec un tableau vide
    for _ in 1:max_length
        push!(transposed_list, [])
    end

    # Itérer sur chaque liste dans list_of_lists
    for sublist in list_of_lists
        # Itérer sur chaque élément dans la sous-liste
        for (i, item) in enumerate(sublist)
            # Ajouter l'élément à la colonne correspondante
            push!(transposed_list[i], item)
        end
    end
    return transposed_list
end


list_of_lists = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
transposed_list = transpose_list_of_lists(list_of_lists)
println(transposed_list)


In [ ]:
# Exemple d'un Vector{Any}
vector_any = [1, 200, 3, 4, 5]

# Convertir en Vector{UInt8}
vector_uint8 = convert(Vector{UInt8}, vector_any)

# Afficher le résultat
println(vector_uint8)
